# 10 — Statistical Hypothesis Testing

**Hypothesis:** Graph topological features significantly improve DDI type prediction, and the predictive power depends on the scope of the graph (full network vs domain-specific subgraph).

**Sub-claims:**
1. Graph features are predictive (RF >> random baseline)
2. Full graph features > oncology subgraph features
3. Liquid cancer and solid cancer subgraphs behave differently

**Statistical methods:**
- **Bootstrap confidence intervals** (1000 resamples) for macro-F1 at each graph level
- **Permutation test** for pairwise model comparisons on shared test edges
- All experiments use **inductive setting** (test edges removed before feature computation)


In [ ]:
import pandas as pd
import numpy as np
import re
import time
import networkx as nx
from collections import Counter
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)


## 1. Setup

In [ ]:
# Drug categories
LIQUID_DRUGS = {
    "Imatinib", "Dasatinib", "Nilotinib", "Bosutinib", "Ponatinib",
    "Daunorubicin", "Idarubicin", "Cytarabine", "Mitoxantrone",
    "Ibrutinib", "Idelalisib", "Venetoclax",
    "Bortezomib", "Carfilzomib", "Ixazomib",
    "Thalidomide", "Lenalidomide", "Pomalidomide",
    "Vincristine", "Vinblastine", "Cyclophosphamide", "Chlorambucil",
    "Mechlorethamine", "Melphalan", "Busulfan",
    "Bleomycin", "Etoposide", "Procarbazine",
    "Ruxolitinib", "Baricitinib", "Tofacitinib",
    "Methotrexate", "Carmustine", "Lomustine", "Fludarabine",
}

SOLID_DRUGS = {
    "Tamoxifen", "Fulvestrant", "Anastrozole", "Letrozole", "Exemestane",
    "Lapatinib", "Neratinib", "Tucatinib",
    "Palbociclib", "Ribociclib", "Abemaciclib",
    "Paclitaxel", "Docetaxel", "Doxorubicin", "Capecitabine",
    "Erlotinib", "Gefitinib", "Afatinib", "Osimertinib", "Icotinib",
    "Crizotinib", "Ceritinib", "Alectinib", "Brigatinib",
    "Vemurafenib", "Dabrafenib", "Encorafenib",
    "Fluorouracil", "Irinotecan", "Oxaliplatin", "Regorafenib",
    "Sorafenib", "Sunitinib", "Pazopanib", "Axitinib", "Cabozantinib", "Lenvatinib",
    "Everolimus", "Temsirolimus",
    "Cisplatin", "Carboplatin", "Olaparib", "Rucaparib", "Niraparib",
    "Bicalutamide", "Flutamide", "Enzalutamide", "Abiraterone",
    "Diethylstilbestrol", "Degarelix", "Goserelin", "Leuprolide",
    "Gemcitabine", "Temozolomide", "Dacarbazine", "Topotecan",
    "Ifosfamide", "Altretamine", "Estramustine", "Thiotepa",
    "Mitomycin", "Teniposide", "Vinorelbine", "Vandetanib", "Epirubicin",
}

ALL_ONCOLOGY = LIQUID_DRUGS | SOLID_DRUGS

PATTERNS = [
    (r"metabolism.*(?:increased|decreased)", "metabolism"),
    (r"active metabolites", "metabolism"),
    (r"serum concentration.*(?:increased|decreased|reduced)", "concentration"),
    (r"bioavailability.*(?:increased|decreased)", "concentration"),
    (r"protein binding", "concentration"),
    (r"risk or severity of adverse effects", "adverse_effects"),
    (r"risk or severity of.*(?:prolongation|bleeding|hypotension|hypertension|heart failure|hyperkalemia)", "adverse_effects"),
    (r"risk of a hypersensitivity", "adverse_effects"),
    (r"therapeutic efficacy.*(?:decreased|increased)", "efficacy"),
    (r"decrease effectiveness", "efficacy"),
    (r"decrease in the absorption", "absorption"),
    (r"absorption.*(?:increased|decreased)", "absorption"),
    (r"excretion rate", "excretion"),
    (r"(?:increase|decrease).*(?:activities|activity)", "activity"),
]

def extract_label(desc):
    for p, l in PATTERNS:
        if re.search(p, desc.lower()): return l
    return None

FEATURE_COLS = [
    "out_degree_u", "in_degree_u", "betweenness_u", "clustering_u", "pagerank_u",
    "out_degree_v", "in_degree_v", "betweenness_v", "clustering_v", "pagerank_v",
    "common_neighbors", "jaccard", "same_community", "degree_diff",
]

def compute_features_on_graph(G, edges):
    G_und = G.to_undirected()
    in_deg = dict(G.in_degree())
    out_deg = dict(G.out_degree())
    betweenness = nx.betweenness_centrality(G, k=min(50, G.number_of_nodes()), seed=42)
    clustering = nx.clustering(G_und)
    pagerank = nx.pagerank(G)
    from networkx.algorithms.community import louvain_communities
    comms = louvain_communities(G_und, seed=42)
    comm_map = {}
    for idx, comm in enumerate(comms):
        for node in comm:
            comm_map[node] = idx
    neighbor_sets = {n: set(G_und.neighbors(n)) for n in G_und.nodes()}
    
    records = []
    for u, v, label in edges:
        if u not in G.nodes() or v not in G.nodes():
            continue
        cn = len(neighbor_sets.get(u, set()) & neighbor_sets.get(v, set()))
        union = len(neighbor_sets.get(u, set()) | neighbor_sets.get(v, set()))
        jc = cn / union if union > 0 else 0.0
        sc = int(comm_map.get(u, -1) == comm_map.get(v, -2))
        records.append({
            "drug_u": u, "drug_v": v, "label": label,
            "out_degree_u": out_deg.get(u, 0), "in_degree_u": in_deg.get(u, 0),
            "betweenness_u": betweenness.get(u, 0), "clustering_u": clustering.get(u, 0),
            "pagerank_u": pagerank.get(u, 0),
            "out_degree_v": out_deg.get(v, 0), "in_degree_v": in_deg.get(v, 0),
            "betweenness_v": betweenness.get(v, 0), "clustering_v": clustering.get(v, 0),
            "pagerank_v": pagerank.get(v, 0),
            "common_neighbors": cn, "jaccard": jc, "same_community": sc,
            "degree_diff": abs((in_deg.get(u,0)+out_deg.get(u,0))-(in_deg.get(v,0)+out_deg.get(v,0))),
        })
    return pd.DataFrame(records)

print("Setup complete.")


## 2. Load Data & Build Subsets

In [ ]:
df_full = pd.read_csv("../data/drug_interactions.csv")
df_full['label'] = df_full['Interaction Description'].apply(extract_label)
df_full = df_full.dropna(subset=['label'])

all_drugs = set(df_full['Drug 1'].unique()) | set(df_full['Drug 2'].unique())
liquid_found = LIQUID_DRUGS & all_drugs
solid_found = SOLID_DRUGS & all_drugs
onc_found = ALL_ONCOLOGY & all_drugs

df_onc = df_full[(df_full['Drug 1'].isin(onc_found)) | (df_full['Drug 2'].isin(onc_found))].copy()
df_liquid = df_full[(df_full['Drug 1'].isin(liquid_found)) | (df_full['Drug 2'].isin(liquid_found))].copy()
df_solid = df_full[(df_full['Drug 1'].isin(solid_found)) | (df_full['Drug 2'].isin(solid_found))].copy()

datasets = {
    'Full Graph': df_full,
    'Oncology': df_onc,
    'Liquid Cancer': df_liquid,
    'Solid Cancer': df_solid,
}

for name, df in datasets.items():
    print(f"{name:<16}: {len(df):>8,} edges")


## 3. Run Inductive RF Experiments

For each graph level:
1. Split edges into 80% train / 20% test
2. Build graph from **training edges only**
3. Compute features on training graph
4. Train RF, predict test edges
5. Save per-sample predictions for statistical testing


In [ ]:
predictions = {}  # Store per-sample results for each graph level

for name, df in datasets.items():
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    
    # Step 1: Split edges FIRST
    edges = list(zip(df['Drug 1'], df['Drug 2'], df['label']))
    labels = [e[2] for e in edges]
    train_edges, test_edges = train_test_split(
        edges, test_size=0.2, random_state=42, stratify=labels
    )
    print(f"  Train: {len(train_edges):,}  Test: {len(test_edges):,}")
    
    # Step 2: Build training-only graph
    t0 = time.time()
    G_train = nx.DiGraph()
    for u, v, label in train_edges:
        G_train.add_edge(u, v, label=label)
    
    # Step 3: Compute features on training graph for BOTH train and test
    df_train_feat = compute_features_on_graph(G_train, train_edges)
    df_test_feat = compute_features_on_graph(G_train, test_edges)
    print(f"  Features computed in {time.time()-t0:.1f}s")
    
    # Step 4: Train RF
    clf = RandomForestClassifier(
        n_estimators=200, max_depth=None, random_state=42,
        class_weight="balanced", n_jobs=-1
    )
    clf.fit(df_train_feat[FEATURE_COLS].values, df_train_feat['label'].values)
    
    y_true = df_test_feat['label'].values
    y_pred = clf.predict(df_test_feat[FEATURE_COLS].values)
    
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    print(f"  RF Macro-F1 (inductive): {macro_f1:.4f}")
    
    # Save predictions
    predictions[name] = {
        'y_true': y_true,
        'y_pred': y_pred,
        'macro_f1': macro_f1,
        'test_pairs': list(zip(df_test_feat['drug_u'], df_test_feat['drug_v'])),
    }
    
    del G_train, df_train_feat, df_test_feat, clf
    import gc; gc.collect()

print("\n\nAll experiments complete.")


## 4. Bootstrap Confidence Intervals

For each graph level, resample the test set predictions 1000 times and compute macro-F1 each time. This gives us a distribution of macro-F1 scores, from which we derive 95% confidence intervals.


In [ ]:
def bootstrap_f1(y_true, y_pred, n_bootstrap=1000, ci=0.95, seed=42):
    """Compute bootstrap confidence interval for macro-F1."""
    rng = np.random.RandomState(seed)
    n = len(y_true)
    f1_scores = []
    
    for _ in range(n_bootstrap):
        idx = rng.choice(n, size=n, replace=True)
        f1_b = f1_score(y_true[idx], y_pred[idx], average='macro', zero_division=0)
        f1_scores.append(f1_b)
    
    f1_scores = np.array(f1_scores)
    alpha = (1 - ci) / 2
    ci_lower = np.percentile(f1_scores, alpha * 100)
    ci_upper = np.percentile(f1_scores, (1 - alpha) * 100)
    
    return {
        'mean': np.mean(f1_scores),
        'std': np.std(f1_scores),
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'scores': f1_scores,
    }


# Run bootstrap for each graph level
print("Running bootstrap (1000 resamples per graph level)...")
bootstrap_results = {}

for name in ['Full Graph', 'Oncology', 'Liquid Cancer', 'Solid Cancer']:
    p = predictions[name]
    bs = bootstrap_f1(p['y_true'], p['y_pred'])
    bootstrap_results[name] = bs
    print(f"  {name:<16}: F1 = {p['macro_f1']:.4f}  95% CI = [{bs['ci_lower']:.4f}, {bs['ci_upper']:.4f}]")


### 4.1 Bootstrap Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']
graph_names = ['Full Graph', 'Oncology', 'Liquid Cancer', 'Solid Cancer']

for ax, name, color in zip(axes.flat, graph_names, colors):
    bs = bootstrap_results[name]
    ax.hist(bs['scores'], bins=40, color=color, alpha=0.7, edgecolor='white')
    ax.axvline(predictions[name]['macro_f1'], color='black', linewidth=2,
              linestyle='-', label=f"Observed: {predictions[name]['macro_f1']:.4f}")
    ax.axvline(bs['ci_lower'], color='red', linewidth=1.5, linestyle='--',
              label=f"95% CI: [{bs['ci_lower']:.4f}, {bs['ci_upper']:.4f}]")
    ax.axvline(bs['ci_upper'], color='red', linewidth=1.5, linestyle='--')
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Macro-F1')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)

plt.suptitle('Bootstrap Distributions of Macro-F1 (RF, Inductive Setting)',
            fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig("../data/viz_bootstrap_distributions.png", dpi=200, bbox_inches='tight')
plt.show()


### 4.2 Confidence Interval Comparison (Presentation Slide)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

graph_names = ['Full Graph', 'Oncology', 'Liquid Cancer', 'Solid Cancer']
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']
y_pos = np.arange(len(graph_names))

for i, (name, color) in enumerate(zip(graph_names, colors)):
    bs = bootstrap_results[name]
    f1 = predictions[name]['macro_f1']
    ci_low = bs['ci_lower']
    ci_high = bs['ci_upper']
    
    # Error bar
    ax.errorbar(f1, i, xerr=[[f1 - ci_low], [ci_high - f1]],
               fmt='o', color=color, markersize=10, capsize=8,
               capthick=2, elinewidth=2, markeredgecolor='white',
               markeredgewidth=1.5)
    
    # Annotation
    ax.text(ci_high + 0.01, i, f'{f1:.4f} [{ci_low:.4f}, {ci_high:.4f}]',
           va='center', fontsize=10)

ax.set_yticks(y_pos)
ax.set_yticklabels(graph_names, fontsize=12)
ax.set_xlabel('Macro-F1 Score', fontsize=13)
ax.set_title('RF Macro-F1 with 95% Bootstrap CI (Inductive Setting)',
            fontsize=14, fontweight='bold')
ax.axvline(1/7, color='gray', linestyle='--', alpha=0.5, label='Random baseline (1/7)')
ax.set_xlim(0, 1.05)
ax.legend(fontsize=10)
ax.grid(axis='x', alpha=0.3)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig("../data/viz_ci_comparison.png", dpi=200, bbox_inches='tight')
plt.show()
print("Saved: viz_ci_comparison.png")


## 5. Permutation Test: Full Graph vs Subgraph

To test whether the full graph RF is **significantly** better than the subgraph RF, we use a permutation test on shared test edges.

**Procedure:**
1. Find drug pairs that appear in both the full graph test set and the subgraph test set
2. Compare per-sample correctness (correct=1, wrong=0) from each model
3. Permutation test: randomly shuffle which model's predictions are assigned to which condition
4. Compute p-value: fraction of permutations where the shuffled difference ≥ observed difference


In [ ]:
def permutation_test_accuracy(correct_A, correct_B, n_perm=10000, seed=42):
    """
    Permutation test comparing two models' per-sample correctness.
    correct_A, correct_B: boolean arrays (True=correct, False=wrong) on SAME samples.
    Tests H0: accuracy_A = accuracy_B.
    Returns observed difference and p-value.
    """
    rng = np.random.RandomState(seed)
    
    obs_diff = correct_A.mean() - correct_B.mean()
    
    combined = np.stack([correct_A, correct_B], axis=1)
    n = len(combined)
    
    count = 0
    for _ in range(n_perm):
        # For each sample, randomly swap which model gets credit
        swap = rng.random(n) > 0.5
        perm_A = np.where(swap, correct_B, correct_A)
        perm_B = np.where(swap, correct_A, correct_B)
        perm_diff = perm_A.mean() - perm_B.mean()
        
        if abs(perm_diff) >= abs(obs_diff):
            count += 1
    
    p_value = count / n_perm
    return obs_diff, p_value


In [ ]:
# Find shared test pairs between Full Graph and each subgraph
comparisons = [
    ('Full Graph', 'Oncology'),
    ('Full Graph', 'Liquid Cancer'),
    ('Full Graph', 'Solid Cancer'),
    ('Liquid Cancer', 'Solid Cancer'),
]

print("=" * 70)
print("PERMUTATION TESTS (10,000 permutations)")
print("=" * 70)

perm_results = {}

for name_a, name_b in comparisons:
    # Find shared test pairs
    pairs_a = set(predictions[name_a]['test_pairs'])
    pairs_b = set(predictions[name_b]['test_pairs'])
    shared = pairs_a & pairs_b
    
    if len(shared) < 10:
        print(f"\n{name_a} vs {name_b}: Only {len(shared)} shared pairs — skipping")
        continue
    
    # Get predictions for shared pairs
    # Build lookup: pair → (true, pred)
    lookup_a = {}
    for i, pair in enumerate(predictions[name_a]['test_pairs']):
        if pair in shared:
            lookup_a[pair] = (predictions[name_a]['y_true'][i], predictions[name_a]['y_pred'][i])
    
    lookup_b = {}
    for i, pair in enumerate(predictions[name_b]['test_pairs']):
        if pair in shared:
            lookup_b[pair] = (predictions[name_b]['y_true'][i], predictions[name_b]['y_pred'][i])
    
    # Align
    shared_list = sorted(shared)
    correct_a = np.array([lookup_a[p][0] == lookup_a[p][1] for p in shared_list])
    correct_b = np.array([lookup_b[p][0] == lookup_b[p][1] for p in shared_list])
    
    acc_a = correct_a.mean()
    acc_b = correct_b.mean()
    
    obs_diff, p_value = permutation_test_accuracy(correct_a, correct_b)
    
    perm_results[(name_a, name_b)] = {
        'n_shared': len(shared),
        'acc_a': acc_a, 'acc_b': acc_b,
        'obs_diff': obs_diff, 'p_value': p_value,
    }
    
    sig = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "n.s."
    print(f"\n{name_a} vs {name_b}:")
    print(f"  Shared test pairs: {len(shared):,}")
    print(f"  {name_a} accuracy: {acc_a:.4f}")
    print(f"  {name_b} accuracy: {acc_b:.4f}")
    print(f"  Observed difference: {obs_diff:+.4f}")
    print(f"  p-value: {p_value:.4f} {sig}")


## 6. McNemar's Test

McNemar's test specifically tests whether two classifiers disagree in a systematically biased way. It focuses on the **discordant pairs** — cases where one model is correct and the other is wrong.


In [ ]:
from scipy.stats import binom_test

def mcnemar_test(correct_a, correct_b):
    """
    McNemar's test for paired classifier comparison.
    Returns: (b, c, chi2, p_value)
    where b = A correct & B wrong, c = A wrong & B correct
    """
    b = ((correct_a) & (~correct_b)).sum()   # A right, B wrong
    c = ((~correct_a) & (correct_b)).sum()   # A wrong, B right
    
    # Use exact binomial test for small counts
    n = b + c
    if n == 0:
        return b, c, 0, 1.0
    
    # Two-sided binomial test: H0: b = c
    p_value = binom_test(b, n, 0.5)
    
    # Chi-squared approximation
    chi2 = (abs(b - c) - 1)**2 / (b + c) if (b + c) > 0 else 0
    
    return b, c, chi2, p_value


print("=" * 70)
print("McNEMAR'S TEST")
print("=" * 70)

for name_a, name_b in comparisons:
    pairs_a = set(predictions[name_a]['test_pairs'])
    pairs_b = set(predictions[name_b]['test_pairs'])
    shared = pairs_a & pairs_b
    
    if len(shared) < 10:
        print(f"\n{name_a} vs {name_b}: Insufficient shared pairs — skipping")
        continue
    
    lookup_a = {}
    for i, pair in enumerate(predictions[name_a]['test_pairs']):
        if pair in shared:
            lookup_a[pair] = (predictions[name_a]['y_true'][i], predictions[name_a]['y_pred'][i])
    lookup_b = {}
    for i, pair in enumerate(predictions[name_b]['test_pairs']):
        if pair in shared:
            lookup_b[pair] = (predictions[name_b]['y_true'][i], predictions[name_b]['y_pred'][i])
    
    shared_list = sorted(shared)
    correct_a = np.array([lookup_a[p][0] == lookup_a[p][1] for p in shared_list])
    correct_b = np.array([lookup_b[p][0] == lookup_b[p][1] for p in shared_list])
    
    b, c, chi2, p = mcnemar_test(correct_a, correct_b)
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."
    
    print(f"\n{name_a} vs {name_b} ({len(shared):,} shared pairs):")
    print(f"  {name_a} correct, {name_b} wrong: {b:>5}")
    print(f"  {name_a} wrong, {name_b} correct: {c:>5}")
    print(f"  McNemar chi²: {chi2:.2f}")
    print(f"  p-value: {p:.6f} {sig}")
    
    if b > c:
        print(f"  → {name_a} is significantly better" if p < 0.05 else f"  → No significant difference")
    else:
        print(f"  → {name_b} is significantly better" if p < 0.05 else f"  → No significant difference")


## 7. Complete Results Summary (Presentation Ready)

In [ ]:
print("=" * 80)
print("COMPLETE RESULTS TABLE")
print("=" * 80)
print(f"{'Graph Level':<18} {'Macro-F1':>10} {'95% CI Lower':>14} {'95% CI Upper':>14} {'Test Edges':>12}")
print("-" * 70)

for name in ['Full Graph', 'Oncology', 'Liquid Cancer', 'Solid Cancer']:
    f1 = predictions[name]['macro_f1']
    bs = bootstrap_results[name]
    n = len(predictions[name]['y_true'])
    print(f"{name:<18} {f1:>10.4f} {bs['ci_lower']:>14.4f} {bs['ci_upper']:>14.4f} {n:>12,}")

print(f"\n{'Random baseline':<18} {1/7:>10.4f}")

print("\n\n" + "=" * 80)
print("PAIRWISE SIGNIFICANCE TESTS")
print("=" * 80)
print(f"{'Comparison':<35} {'Diff':>8} {'p-value':>10} {'Sig':>5}")
print("-" * 60)

for (na, nb), r in perm_results.items():
    sig = "***" if r['p_value'] < 0.001 else "**" if r['p_value'] < 0.01 else "*" if r['p_value'] < 0.05 else "n.s."
    print(f"{na + ' vs ' + nb:<35} {r['obs_diff']:>+8.4f} {r['p_value']:>10.4f} {sig:>5}")


### 7.1 Final Comparison with Significance Annotations

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

graph_names = ['Full Graph', 'Oncology', 'Liquid Cancer', 'Solid Cancer']
f1_scores = [predictions[n]['macro_f1'] for n in graph_names]
ci_lows = [bootstrap_results[n]['ci_lower'] for n in graph_names]
ci_highs = [bootstrap_results[n]['ci_upper'] for n in graph_names]
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

x = np.arange(len(graph_names))
bars = ax.bar(x, f1_scores, color=colors, alpha=0.85, edgecolor='white', linewidth=2)

# Error bars (CI)
for i, (f1, ci_l, ci_h, color) in enumerate(zip(f1_scores, ci_lows, ci_highs, colors)):
    ax.errorbar(i, f1, yerr=[[f1 - ci_l], [ci_h - f1]],
               fmt='none', color='black', capsize=8, capthick=2, elinewidth=2)
    ax.text(i, ci_h + 0.02, f'{f1:.3f}\n[{ci_l:.3f}, {ci_h:.3f}]',
           ha='center', va='bottom', fontsize=10, fontweight='bold')

# Significance brackets (if we have results)
if ('Full Graph', 'Oncology') in perm_results:
    r = perm_results[('Full Graph', 'Oncology')]
    sig = "***" if r['p_value'] < 0.001 else "**" if r['p_value'] < 0.01 else "*" if r['p_value'] < 0.05 else "n.s."
    y_bracket = max(f1_scores[0], f1_scores[1]) + 0.08
    ax.plot([0, 0, 1, 1], [y_bracket-0.01, y_bracket, y_bracket, y_bracket-0.01], 'k-', linewidth=1.5)
    ax.text(0.5, y_bracket + 0.005, f'p={r["p_value"]:.4f} {sig}', ha='center', fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(graph_names, fontsize=12)
ax.set_ylabel('Macro-F1 Score', fontsize=13)
ax.set_title('RF Macro-F1 Across Graph Levels\n(Inductive Setting, 95% Bootstrap CI)',
            fontsize=14, fontweight='bold')
ax.axhline(y=1/7, color='gray', linestyle='--', alpha=0.4, label='Random baseline')
ax.set_ylim(0, 1.15)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("../data/viz_final_results_with_ci.png", dpi=200, bbox_inches='tight')
plt.show()
print("Saved: viz_final_results_with_ci.png")


## 8. Hypothesis Verdict

### Hypothesis: Graph topological features significantly improve DDI type prediction, and the predictive power depends on the scope of the graph.

**Claim 1: Graph features are predictive**
→ All RF macro-F1 scores are significantly above the random baseline (1/7 ≈ 0.143). **SUPPORTED.**

**Claim 2: Full graph > subgraph**
→ Check if the CI for Full Graph is strictly above the CI for Oncology/Liquid/Solid. Check permutation test p-values. **See results above.**

**Claim 3: Liquid vs Solid behave differently**
→ Check if their CIs overlap and permutation test result. **See results above.**
